In [ ]:
import copy
import random
import gymnasium as gym
import numpy as np
from datetime import datetime, timedelta
import pickle
import pandas as pd
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import imageio.v2 as imageio

In [ ]:
class Config():
    def __init__(self):
        # env
        self.env_name = "Taxi-v4"
        self.gamma = 0.99
        self.num_action = 6
        self.state_dim = 1

        # training
        self.total_episodes = None
        self.total_steps = 100000
        self.learning_rate = 5e-5
        self.max_gradient_norm = 0.5
        self.normalize_advantage = False
        self.num_envs = 16

        self.value_coef = 0.5
        self.entropy_coef = 0.02

        self.n_steps = 32
        self.test_frequency = 1000 # 1000
        self.save_frequency = 100000000 # Don't save

        self.seed = 1
        self.test_seeds = [
    1001, 1003, 1007, 1009, 1013, 1019, 1021, 1031, 1033, 1039,
    1049, 1051, 1061, 1063, 1069, 1087, 1091, 1093, 1097, 1103,
    1109, 1117, 1123, 1129, 1151, 1153, 1163, 1171, 1181, 1187,
    1193, 1201, 1213, 1217, 1223, 1229, 1231, 1237, 1249, 1259,
    1277, 1279, 1283, 1289, 1291, 1297, 1301, 1303, 1307, 1319,
    1321, 1327, 1361, 1367, 1373, 1381, 1399, 1409, 1423, 1427,
    1429, 1433, 1439, 1447, 1451, 1453, 1459, 1471, 1481, 1483,
    1487, 1489, 1493, 1499, 1511, 1523, 1531, 1543, 1549, 1553,
    1559, 1567, 1571, 1579, 1583, 1597, 1601, 1607, 1609, 1613,
    1619, 1621, 1627, 1637, 1657, 1663, 1667, 1669, 1693, 1697
]
        #[1, 2, 3, 5, 7, 11, 13, 17, 19, 20]
        self.save_file = "weights/A2C-Taxi-v4-1M-run1"

config = Config()

In [ ]:
random.seed(config.seed)
np.random.seed(config.seed)

In [ ]:
class Replay_Buffer():
    def __init__(self):
        self.reset()
        
    def save(self, data):
        self.states.append(data[0])
        self.actions.append(data[1])
        self.rewards.append(data[2])
        self.next_states.append(data[3])
        self.dones.append(data[4])
        self.logits.append(data[5])
        self.values.append(data[6])

    def reset(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.dones = []
        self.logits = []
        self.values = []

    def get(self):
        return torch.as_tensor(np.array(self.states)), torch.as_tensor(np.array(self.actions)), \
               torch.as_tensor(np.array(self.rewards)), torch.as_tensor(np.array(self.next_states)), \
               torch.as_tensor(np.array(self.dones, dtype = np.float32)), torch.stack(self.logits, 0), torch.stack(self.values, 0)

In [ ]:
class Model(nn.Module):
    def __init__(self, state_dim=4, num_action=2):
        super(Model, self).__init__()
        self.state_dim = state_dim
        self.fc_p = nn.Linear(state_dim, 64)
        self.policy = nn.Linear(64, num_action)

        self.fc_v = nn.Linear(state_dim, 64)
        self.value = nn.Linear(64, 1)

    def forward(self, state):
        x = F.one_hot(state.long(), 500).float().reshape(-1, self.state_dim)
        logits = F.relu(self.fc_p(x))
        logits = self.policy(logits)

        values = F.relu(self.fc_v(x))
        values = self.value(values)
        return logits, values

In [ ]:
class Agent():
    def __init__(self, model, replay_buffer, envs, config):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config.learning_rate, betas=(0.9, 0.999))

        self.replay_buffer = replay_buffer

        self.env_name = config.env_name
        self.gamma = config.gamma
        self.num_action = config.num_action
        self.state_dim = config.state_dim

        # training
        self.total_episodes = config.total_episodes
        self.total_steps = config.total_steps
        self.learning_rate = config.learning_rate
        self.max_gradient_norm = config.max_gradient_norm
        self.normalize_advantage = config.normalize_advantage
        self.num_envs = config.num_envs

        self.value_coef = config.value_coef
        self.entropy_coef = config.entropy_coef

        self.n_steps = config.n_steps
        self.test_frequency = config.test_frequency
        self.save_frequency = config.save_frequency

        self.seed = config.seed
        self.test_seeds = config.test_seeds
        self.save_file = config.save_file

        self.test_env = gym.make(self.env_name, is_rainy=False, fickle_passenger=False)
        self.envs = envs

    def sample_action(self, state, is_testing = False):
        if is_testing:
            with torch.no_grad():
                state = torch.tensor(np.array(state), device = self.device).unsqueeze(0)
                logits, values = self.model(state)
                action = logits[0].argmax(-1).item()
        else:
            state = torch.tensor(np.array(state), device = self.device).unsqueeze(1)
            logits, values = self.model(state)
            with torch.no_grad():
                probs = torch.softmax(logits, -1)
                distribution = torch.distributions.Categorical(probs)
                action = distribution.sample().cpu().numpy()
        return action, logits, values
    
    def train(self):
        #nxe    nxe      nxe      nxe          nxe    nxexa   nxex1
        states, actions, rewards, next_states, dones, logits, values = self.replay_buffer.get()
        
        with torch.no_grad():
            _, next_values = self.model(next_states[-1].unsqueeze(-1))
            next_values = next_values

        target = next_values.detach().reshape(-1)
        targets = []
        for t in range(self.n_steps-1, -1, -1):
            target = rewards[t] + target * self.gamma * (1.-dones[t])
            targets.append(target)

        targets = torch.stack(targets[::-1]).reshape(-1)
        advantages = targets - values.detach().reshape(-1)

        if self.normalize_advantage:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-9)

        #disks = torch.distributions.Categorical(logits=logits)
        probs = torch.softmax(logits.reshape(self.n_steps * self.num_envs, -1), -1)

        index = torch.arange(0, len(probs), device = self.device)

        entropy = - (probs*(probs + 1e-9).log()).sum(-1).mean()
        loss_p = -((probs[index, actions.reshape(-1)] + 1e-9).log() * advantages).mean()
        loss_v = F.mse_loss(values.reshape(-1), targets)
        loss = loss_p + self.value_coef * loss_v - self.entropy_coef * entropy
        #loss = -(disks.log_prob(actions) * returns).mean()

        self.optimizer.zero_grad()
        loss.backward()
        if self.max_gradient_norm is not None:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_gradient_norm)
        self.optimizer.step()
        return loss_p.item(), loss_v.item(), entropy.item(), loss.item()

    def test(self):
        test_episode_rewards = []
        test_episode_steps = []

        for test_seed in self.test_seeds:
            state = self.test_env.reset(seed=test_seed)[0]
            self.test_env.action_space.seed(test_seed)
            done = False
            episode_reward = 0.
            episode_step = 0

            while not done:
                action, _, _ = self.sample_action(state, True)
                next_state, reward, terminated, truncated, info = self.test_env.step(action)
                episode_step += 1
                episode_reward += reward
                done = terminated or truncated
                state = next_state

            test_episode_steps.append(episode_step)
            test_episode_rewards.append(episode_reward)
        return test_episode_rewards, test_episode_steps
    
    def render(self, seed):
        self.model.load_state_dict(torch.load(f"{self.save_file}.pth"))

        self.render_env = gym.make(self.env_name, is_rainy=False, fickle_passenger=False, render_mode='human')
        state = self.render_env.reset(seed=seed)[0]
        self.render_env.action_space.seed(seed)
        done = False
        episode_reward = 0.

        while not done:
            time.sleep(0.1)
            action, _, _ = self.sample_action(state, True)
            next_state, reward, terminated, truncated, info = self.render_env.step(action)
            episode_reward += reward
            done = terminated or truncated
            state = next_state
        self.render_env.close()
        print(episode_reward)

    def save_gif(self, seed):
        self.model.load_state_dict(torch.load(f"{self.save_file}.pth"))

        self.render_env = gym.make(self.env_name, is_rainy=False, fickle_passenger=False, render_mode="rgb_array")
        state = self.render_env.reset(seed=seed)[0]
        self.render_env.action_space.seed(seed)
        done = False
        episode_reward = 0.
        frames = [self.render_env.render()]

        while not done:
            action, _, _ = self.sample_action(state, True)
            next_state, reward, terminated, truncated, info = self.render_env.step(action)
            frames.append(self.render_env.render())
            episode_reward += reward
            done = terminated or truncated
            state = next_state
        self.render_env.close()
        print(episode_reward)

        imageio.mimsave(
            f"{self.save_file}.gif",
            frames,
            fps=60,
            loop=0
        )

    def learn(self):
        self.current_step = 0

        self.max_test_reward = -1000.
        self.episode_rewards = []
        self.episode_steps = []
        self.max_episode_rewards = -1000.
        self.test_episode_rewards = []
        self.test_episode_steps = []
        self.losses = []
        self.losses_p, self.losses_v, self.losses_e = [], [], []

        start_time = datetime.now()
        start_episode_times = [datetime.now() for _ in range(self.num_envs)]
        states = self.envs.reset(seed=self.seed)[0]
        self.envs.action_space.seed(self.seed)
        done = False
        episode_rewards = np.zeros((self.num_envs,))
        episode_steps = np.zeros((self.num_envs,))
        episode = 0

        while True:
            episode_steps += 1
            self.current_step += 1
            if self.total_episodes is not None and episode > self.total_episodes:
                break
            if self.current_step > self.total_steps:
                break

            actions, logits, values = self.sample_action(states)
            next_states, rewards, terminations, truncations, infos = self.envs.step(actions)
            episode_rewards += np.array(rewards)
            dones = (np.array(terminations) | np.array(truncations))
            
            self.replay_buffer.save((states, actions, rewards, next_states, dones, logits, values))

            states = next_states

            for env_id, done in enumerate(dones):
                if done:
                    episode += 1
                    self.episode_rewards.append(episode_rewards[env_id])
                    self.episode_steps.append(episode_steps[env_id])

                    self.max_episode_rewards = max(self.max_episode_rewards, episode_rewards[env_id])

                    if episode % 10 == 0:
                        print(f"Step: {self.current_step}, Episode: {episode} Env_id: {env_id} "
                            f"Steps: {episode_steps[env_id]}, Rewards: {episode_rewards[env_id]}, "
                            f"Mean_Rewards: {np.array(self.episode_rewards[-min(100, len(self.episode_rewards)):]).mean():.2f}±"
                            f"{np.array(self.episode_rewards[-min(100, len(self.episode_rewards)):]).std():.2f}, "
                            f"Min: {np.array(self.episode_rewards[-min(100, len(self.episode_rewards)):]).min():.2f}, "
                            f"Max: {np.array(self.episode_rewards[-min(100, len(self.episode_rewards)):]).max():.2f}, "
                            f"Max_Rewards: {self.max_episode_rewards}, Loss: {np.array(self.losses[-min(1000, len(self.losses)):]).mean():.2f}, "
                            f"Loss_P: {np.array(self.losses_p[-min(1000, len(self.losses_p)):]).mean():.2f}, "
                            f"Loss_V: {np.array(self.losses_v[-min(1000, len(self.losses_v)):]).mean():.2f}, "
                            f"Loss_E: {np.array(self.losses_e[-min(1000, len(self.losses_e)):]).mean():.2f}, "
                            f"Duration: {datetime.now() - start_episode_times[env_id]}, Total_Time: {datetime.now() - start_time}")

                    episode_rewards[env_id] = 0
                    episode_steps[env_id] = 0
                    start_episode_times[env_id] = datetime.now()

            if self.current_step % self.n_steps == 0:
                loss_p, loss_v, entropy, loss = self.train()
                self.losses.append(loss)
                self.losses_p.append(loss_p)
                self.losses_v.append(loss_v)
                self.losses_e.append(entropy)
                self.replay_buffer.reset()

            if self.current_step % self.test_frequency == 0:
                start_test_time = datetime.now()

                test_episode_rewards, test_episode_steps = self.test()
                self.test_episode_rewards.append(test_episode_rewards)
                self.test_episode_steps.append(test_episode_steps)

                if self.max_test_reward < np.array(test_episode_rewards).mean():
                    self.max_test_reward = np.array(test_episode_rewards).mean()
                    torch.save(self.model.state_dict(), f"{self.save_file}.pth")

                if len(self.test_episode_rewards) % 10 == 0:
                    print(f"Test Episode: {len(self.test_episode_rewards)} Current_Step: {self.current_step} "
                          f"Rewards: {np.array(test_episode_rewards).mean():.2f}±{np.array(test_episode_rewards).std():.2f} "
                          f"Min: {np.array(test_episode_rewards).min():.2f} Max: {np.array(test_episode_rewards).max():.2f} "
f"Mean_Rewards: {np.array(self.test_episode_rewards[-min(100, len(self.test_episode_rewards)):]).mean():.2f}±"
f"{np.array(self.test_episode_rewards[-min(100, len(self.test_episode_rewards)):]).mean(-1).std():.2f} "
f"Min: {np.array(self.test_episode_rewards[-min(100, len(self.test_episode_rewards)):]).mean(-1).min():.2f} "
f"Max: {np.array(self.test_episode_rewards[-min(100, len(self.test_episode_rewards)):]).mean(-1).max():.2f} Max_Rewards: {self.max_test_reward} \
Time: {datetime.now() - start_test_time} Total_Time: {datetime.now() - start_time}")
            if self.current_step % self.save_frequency == 0:
                torch.save(self.model.state_dict(), f"{self.save_file}_{self.current_step}.pth")

        with open(f"{self.save_file}-log.pickle", "wb") as f:
            pickle.dump({"episode_rewards": self.episode_rewards,
                         "test_episode_rewards": self.test_episode_rewards,
                         "episode_steps": self.episode_steps,
                         "test_episode_steps": self.test_episode_steps,
                         "step": self.current_step}, f)

In [ ]:
replay_buffer = Replay_Buffer()
model = Model(500, config.num_action)
envs = gym.make_vec(config.env_name, num_envs=config.num_envs, vectorization_mode="async")

agent = Agent(model, replay_buffer, envs, config)

In [ ]:
agent.learn()

In [ ]:
agent.render(95)

In [ ]:
agent.save_gif(95)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

total_steps = [int(1e5), int(2e5), int(5e5), int(1e6)]
steps = ["100K", "200K", "500K", "1M"]
steps_frequency = [10, 5, 2, 1]
colors = ["red", "green", "blue", "purple"]

plt.figure(figsize=(10, 6))

for total_step, step, step_frequency, color in zip(total_steps, steps, steps_frequency, colors):

    for run in range(1, 2):
        log_file = f"weights/A2C-Taxi-v4-{step}-run{run}-log.pickle"
        window = 100
        test_frequency = config.test_frequency   

        try:
            with open(log_file, "rb") as f:
                log = pickle.load(f)
        except:
            continue

        episode_rewards = np.array(log["episode_rewards"])
        test_episode_rewards = np.array(log["test_episode_rewards"]).mean(-1)

        train_episode = np.array(log["episode_steps"])
        for i in range(1, len(train_episode)):
            train_episode[i] += train_episode[i-1]
        test_episode = np.arange(config.test_frequency, total_step, config.test_frequency)

    sum_episode_rewards = np.convolve(episode_rewards, np.ones((window,), dtype=np.float32))[:len(episode_rewards)]
    mean_episode_rewards = sum_episode_rewards / np.array([i+1 if i < window else window for i in range(len(episode_rewards))])

    # sum_test_episode_rewards = np.convolve(test_episode_rewards, np.ones((window,), dtype=np.float32))[:len(test_episode_rewards)]
    # mean_test_episode_rewards = sum_test_episode_rewards / np.array([i+1 if i < window else window for i in range(len(test_episode_rewards))])

    plt.plot(
        train_episode * step_frequency,
        mean_episode_rewards,
        label=f"{step} Step Train (Last-{window}-Average)",
        color = color
    )

    plt.plot(
        test_episode * step_frequency,
        test_episode_rewards, #mean_test_episode_rewards,
        linestyle = "--",
        linewidth=2,
        label=f"{step} Step Test", #f"{step} Step Test (Last-{window}-Average)",
        color = color
    )

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Train/Test Reward vs Episode")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
steps = ["100K", "200K", "500K", "1M"]
rewards = [[] for _ in range(len(steps))]

for i, step in enumerate(steps):
    for run in range(1, 11):
        log_file = f"weights/A2C-Taxi-v4-{step}-run{run}-log.pickle"

        try:
            with open(log_file, "rb") as f:
                log = pickle.load(f)
                test_rewards = np.array(log["test_episode_rewards"])
                test_rewards = test_rewards.mean(-1)
                test_rewards = test_rewards.max()
            
                rewards[i].append(test_rewards)
        except:
            continue

rewards = np.array(rewards)

pd.DataFrame(data = {"step": steps, "mean": rewards.mean(-1), "std": rewards.std(-1),
                     "min": rewards.min(-1), "max": rewards.max(-1)})